# Let an OpenAI Model Use Your MCP Tools

Now the fun part: give an LLM your server's tools and let it decide when to call them.

You'll need an API key from the OpenAI platform, set as `OPENAI_API_KEY`.


In [ ]:
# !pip install -q "mcp>=2" openai

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.environ.get("OPENAI_API_KEY"):
    print("Add openai key")

In [2]:
%%writefile server_llm.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Shop")

PRICES = {"apple": 0.5, "banana": 0.25, "mango": 1.75}


@mcp.tool()
def get_price(item: str) -> float:
    """Get the price of one fruit in dollars."""
    return PRICES.get(item.lower(), -1)


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


if __name__ == "__main__":
    mcp.run()

Overwriting server_llm.py


In [3]:
import os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def server_params(script):
    return StdioServerParameters(command=sys.executable, args=[script])


# Jupyter's stderr has no real file descriptor, so we send server logs to devnull
DEVNULL = open(os.devnull, "w")

In [4]:
import json
from openai import OpenAI

MODEL = "gpt-5-mini"
llm = OpenAI()


async def ask(question, script="server_llm.py"):
    async with stdio_client(server_params(script), errlog=DEVNULL) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            # 1. Convert MCP tools into the format the OpenAI API expects
            mcp_tools = (await session.list_tools()).tools
            tools = [
                {
                    "type": "function",
                    "function": {"name": t.name, "description": t.description or "", "parameters": t.input_schema},
                }
                for t in mcp_tools
            ]

            messages = [{"role": "user", "content": question}]

            # 2. Loop: model asks for a tool -> we call it via MCP -> send result back
            while True:
                resp = llm.chat.completions.create(
                    model=MODEL,
                    tools=tools,
                    messages=messages,
                )
                msg = resp.choices[0].message
                messages.append(msg)

                if not msg.tool_calls:
                    return msg.content

                for call in msg.tool_calls:
                    args = json.loads(call.function.arguments)
                    print(f"  calling {call.function.name}({args})")
                    out = await session.call_tool(call.function.name, args)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": call.id,
                            "content": out.content[0].text,
                        }
                    )

In [5]:
answer = await ask(
    "How much do 3 mangos and 4 bananas cost in total? Call the get_price tool to get the price, call add tool to add something, and use multiply tool to multiply something"
)
print("\nANSWER:", answer)

  calling get_price({'item': 'mango'})
  calling get_price({'item': 'banana'})
  calling multiply({'a': 1.75, 'b': 3})
  calling multiply({'a': 0.25, 'b': 4})
  calling add({'a': 5.25, 'b': 1.0})

ANSWER: 3 mangos at $1.75 each = $5.25
4 bananas at $0.25 each = $1.00
Total = $6.25


## What just happened?

1. We asked MCP for the tool list and passed it to the model
2. The model chose tools and arguments
3. **We** executed them through MCP and returned the results
4. The model wrote the final answer

This loop is exactly what apps like ChatGPT, Claude Desktop and Cursor do for you.

## Series wrap-up

You now know the protocol, tools, resources, prompts, clients, and LLM integration.

**Bonus idea:** add your server to an MCP host app (like Claude Desktop or Cursor), then build a knowledge graph memory server.
